## TCDB Composition
This notebook quickly analyzes the composition of TCDB and its transporters to scope in on what kind of transporters to aim for throughout the project.

In [2]:
from Bio import SeqIO
import plotly.express as px
import pandas as pd
import itertools

First up is a sun burst chart of the composition of TCIDs on TCDB, down to family-level.

In [4]:
file_path = "../files/tcdb_all.txt"

tc_data = []
for record in SeqIO.parse(file_path, "fasta"):
    header = record.description
    tc_id = header.split("|")[3]
    parts = tc_id.split(".")
    class_ = parts[0]
    subclass = ".".join(parts[:2])
    family = ".".join(parts[:3])
    tc_data.append({"Class": class_, "Subclass": subclass, "Family": family})

df_all = pd.DataFrame(tc_data)

all_classes = sorted(df_all["Class"].unique())
palette = itertools.cycle(px.colors.qualitative.Set3)
color_map = {cls: next(palette) for cls in all_classes}

fig = px.sunburst(df_all, path=["Class", "Subclass", "Family"], color="Class", color_discrete_map=color_map)
fig.write_image("tcdb_composition.pdf")
fig.show()

Based on the composition of transporters on TCDB shown from the sunburst chart above, classes 1, 2 and 3 are the most prominent, and will be delved more into, througout the project. Class 9 is a temporary class for putative transporters, which either gets moved or removed upon inspection by the maintaners of TCDB. Class 8 also seems to contain a lot of transporters. Upon inspection o n TCDB, and the mapping file of transporters and their substrates, there are only 129 transproters in class 8.A with substrates. And more than half of these are either just protein, molecule, polypeptide or some vague description. Hence, class 8 can also be discarded.

The subclasses that contain the most transporters, and which will be further scrutinized, are 1.A, 1.B, 1.C, 2.A and 3.A. From these subclasses, the 10 largest families from each will be used.

In [3]:
subfamilies = ['1.A', '1.B', '1.C', '2.A', '3.A']
df1 = df_all[df_all['Subclass'].isin(subfamilies)]

top_families = (
    df1.groupby('Subclass')['Family']
    .value_counts()
    .groupby(level=0)
    .nlargest(10)
    .reset_index(level=0, drop=True)
)

family_list = top_families.index.get_level_values('Family').tolist() # In order to analyze the amount of transporters with a substrate assigned, towards the end

def write_families_to_file(families, file_path):
    with open(file_path, 'w') as f:
        for family in families:
            f.write(family + '\n')

write_families_to_file(family_list, "families_all.txt")


tot = 0
for el in top_families:
    tot += el
print(f"A total of {tot} transporters are included in these families, but not all of them are connected to any substrate.")

A total of 8150 transporters are included in these families, but not all of them are connected to any substrate.


This has now been done with the fasta-file of all transport proteins. Now, let's check if the composition of transporters with known substrates are any different. If so, perhaps the perspectives from that sunburst chart might be different, hence the main families as well? Let's see!

In [4]:
file_path = "../files/tcdb_substrates.txt"
tc_ids = []
with open(file_path, "r") as f:
    for line in f:
        tc_ids.append(line.split("\t")[0])
tc_data = []

for id in tc_ids:
    parts = id.split('.')
    class_ = parts[0]
    subclass = '.'.join(parts[:2])
    family = '.'.join(parts[:3])
    tc_data.append({'Class': class_, 'Subclass': subclass, 'Family': family})
df_subs = pd.DataFrame(tc_data)

fig = px.sunburst(df_subs, path=['Class', 'Subclass', 'Family'], color='Class')
fig.show()

Well, the composition certainly seems a little different. Then, I'll extract the top families of each of the same subclasses, as they still are to be the most prominent.

In [5]:
subfamilies = ['1.A', '1.B', '1.C', '2.A', '3.A']
df1 = df_subs[df_subs['Subclass'].isin(subfamilies)]

top_families = (
    df1.groupby('Subclass')['Family']
    .value_counts()
    .groupby(level=0)
    .nlargest(10)
    .reset_index(level=0, drop=True)
)

tot = 0
for el in top_families:
    tot += el

print(f"A total of {tot} transporters are assigned with substrates from these families.")

A total of 4043 transporters are assigned with substrates from these families.


Now, the families from tcdb_all and corresponding reactions can be found in family_mechanisms_all.txt. Likewise for the substrate specific file. They respectively contain 10 and 8 Nan values for mechanisms. _all contain 8150 transporters, while _substrates contain 4043. Now I am checking how many of _alls transporters have a substrate mapped to itself.

In [6]:
df_subs
count = df_subs[df_subs['Family'].isin(family_list)].shape[0]
count

3909

Of the 8150 transporters from _all, only 3909 had any substrate assigned.  I'll use the online version, so it is always the newest update, and has likely increased since download of the file that was analysed. (Downloaded in January)

It was decided to implement _all, and not _substrates. This better encapsulates the total picture of TCDB.

The sections below are not sections for running any useful code, and will likely yield error messages when running on later occasions, as they only serve as blocks for manipulating .tsv files.

Now trying to create an applicable .tsv, as they are easier to use than txt files. First family_mechanisms_all was created manually. Subsequently families_all and mechanisms_all were created. Now I'll try to merge them nicely. This is because I struggled hard to figure out any way to convert the original family_mechanisms_all to a tsv that was readable.

In [7]:
with open("families_all.txt", "r") as f:
    families = f.read().strip().split("\n")

with open("mechanisms_all.txt", "r") as f:
    mechanisms = f.read().strip().split("\n")

df = pd.DataFrame({"Family": families,"Mechanism": mechanisms})

df.to_csv("All_comp/families_mechanisms_all.tsv", sep="\t", index=False)

FileNotFoundError: [Errno 2] No such file or directory: 'mechanisms_all.txt'

Subsequently, family_mechanisms_entity_all.tsv was created to identify the substrate that is essential for each reaction. From this, a manual effort was done to create the final file, families_mechanisms_entity_final.tsv. This was done as then all transporters get mapped to all the reaction mechanisms, and not just the first one.\
Below is nothing but code for adjusting the tsv with pandas manipulations. This was done as it was not possible to edit the tsv as freely as believed to create new columns and edits...

In [ ]:
df_edit = pd.read_csv("All_comp/families_mechanisms_entity_final.tsv", sep="\t")

# df_edit.loc[df_edit["Family"] == "1.A.9", "Mechanism"] = "ions (in) â‡Œ ions (out)"
# df_edit.loc[df_edit["Family"] == "1.A.9", "Acting Entity"] = "ions"
# n Na+ (in) + ATP â‡Œ n Na+ (out) + ADP + Pi Na+
# df_edit.loc[df_edit["Family"]]
mask = (df_edit["Family"] == "3.A.2") & (df_edit["Mechanism"].isna() | df_edit["Acting Entity"].isna())
ind = mask[mask].index[0]
df_edit.loc[ind, "Mechanism"] = "n Na+ (in) + ATP ⇌ n Na+ (out) + ADP + Pi"
df_edit.loc[ind, "Acting Entity"] = "Na+"
df_edit.to_csv("All_comp/families_mechanisms_entity_final.tsv", sep="\t", index=False)